In [8]:
import numpy as np
from matplotlib import pyplot as plt
from sklearn.decomposition import PCA, FastICA
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, KFold, cross_validate

from scipy.stats import skew, kurtosis

import seaborn as sns
import pandas as pd
import matplotlib.colors as mcolors

from tqdm import tqdm
from os import listdir
from os.path import join
import os

In [9]:
def get_training_data(path:str, n_img=None)->tuple:
    
    cover_path = join(path, "cover")
    stego_path = join(path, "stego")

    list_of_covers = sorted(listdir(cover_path))
    list_of_stegos = sorted(listdir(stego_path))

    if n_img:
        n_img = min([len(list_of_covers), len(list_of_stegos), n_img])
    else:
        n_img = min([len(list_of_covers), len(list_of_stegos)])

    samples = np.zeros((n_img, 2, 2, 4))

    for i in tqdm(range(n_img)):
        tmp = np.load(join(cover_path, list_of_covers[i]))[:,:]
        samples[i,0,:,0] = tmp.mean(axis=0)
        samples[i,0,:,1] = tmp.std(axis=0)
        samples[i,0,:,2] = skew(tmp, axis=0)
        samples[i,0,:,3] = kurtosis(tmp, axis=0)

    for i in tqdm(range(n_img)):
        tmp = np.load(join(stego_path, list_of_stegos[i]))[:,:]
        samples[i,1,:,0] = tmp.mean(axis=0)
        samples[i,1,:,1] = tmp.std(axis=0)
        samples[i,1,:,2] = skew(tmp, axis=0)
        samples[i,1,:,3] = kurtosis(tmp, axis=0)

    samples_training = np.concatenate([samples[:,0], samples[:,1]])
    samples_training = samples_training.reshape(-1,samples_training.shape[1]*samples_training.shape[2])
    labels = np.concatenate([np.zeros((len(list_of_covers))), np.ones((len(list_of_stegos)))])

    return samples_training, labels

In [10]:
base_path = "/data2/antoine/iminim_steg/dataset/COCO/ica_subbands"

samples, labels = get_training_data(base_path)

100%|██████████| 2500/2500 [00:45<00:00, 54.52it/s]


In [ ]:
lr = LogisticRegression(max_iter=5_000)
cv_results = cross_validate(lr, samples,labels, cv=5)

print(cv_results["test_score"], cv_results["test_score"].mean(), cv_results["test_score"].std())